# Tutorial: Trader workflow

Audience:
- Power users who need a fast notebook for day-ahead pricing and generation context.

Prerequisites:
- `luminus-py` installed with notebook dependencies.
- A working `luminus-mcp` install on `PATH`.
- API keys configured for the live market data sources used by your environment.

Learning goals:
- Pull prices across multiple zones in one pass.
- Compare zone-level price bands with a small pandas summary.
- Pull generation mix for context.
- Discover the live tool surface available to the trader profile.


## Outline

1. Set up a trader profile client.
2. Pull day-ahead prices across the core zones.
3. Summarize the price spread by zone.
4. Pull generation mix for context.
5. Inspect the live trader tool surface.
6. Capture a reusable next-step checklist.


In [ ]:
from __future__ import annotations

from luminus import Luminus
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)


def resolve_metric_column(frame: pd.DataFrame, preferred: list[str]) -> str:
    for name in preferred:
        if name in frame.columns:
            return name

    numeric = frame.select_dtypes(include="number").columns.tolist()
    if not numeric:
        raise KeyError("No numeric column found to summarize.")
    return numeric[0]


trader = Luminus(profile="trader")
generation = Luminus(profile="generation")
trader, generation


## Step 1 - Pull prices

Start with a compact multi-zone pull. The helper adds the requested zone back into the result so you can compare zones in one frame.


In [ ]:
zones = ["DE", "FR", "NL"]
prices_df = trader.get_day_ahead_prices_many(zones, parallel=True)

display(prices_df.head(8))
prices_df.columns.tolist()


## Step 2 - Summarize the spread

This summary keeps the notebook resilient if the price column name changes slightly between server builds.


In [ ]:
zone_col = "request_zone" if "request_zone" in prices_df.columns else ("zone" if "zone" in prices_df.columns else None)
price_col = resolve_metric_column(prices_df, ["price_eur_mwh", "price", "value", "day_ahead_price", "eur_mwh", "market_price"])

if zone_col is not None:
    price_summary = (
        prices_df.groupby(zone_col)[price_col]
        .agg(mean="mean", low="min", high="max")
        .round(2)
        .sort_values("mean", ascending=False)
    )
else:
    price_summary = prices_df[[price_col]].agg(["mean", "min", "max"]).T.round(2)

display(price_summary)


## Step 3 - Add generation context

Use the generation profile for the mix pull, then compare that context against the price table above.


In [ ]:
generation_df = generation.get_generation_mix_many(["DE", "FR"], parallel=True)

display(generation_df.head(8))
generation_df.columns.tolist()


## Step 4 - Inspect the live tool surface

The trader profile stays narrow on purpose. Inspect its live surface first, then open a second specialist profile when you need more context.


In [ ]:
trader.list_tools()


## Exercises

- Swap in a different zone list and compare the summary table.
- Add a new metric resolver candidate if your deployment uses a different price column label.
- Re-run the same pattern for a separate trading desk profile.


## Pitfall

Do not assume every deployment uses the same field names. Keep the resolver cell so the notebook stays readable when the server schema changes slightly.


## Extension

If you want a production-ready version, wrap the data pull and summary cells in a small helper that accepts a zone list and returns one tidy table.


In [ ]:
trader.close()
generation.close()
